In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\projetos git\analise-de-times-br\data\raw\campeonato-brasileiro-cartoes.csv")
display(df)

,posicao,num_camisa,clube,minuto,atleta,partida_id,cartao,rodata
0,Zagueiro,13.0,Ceara,78,Luiz Otávio,7034,cartao amarelo,15
1,Meio-campo,5.0,ATLETICO-GO,8,Willian Maranhão,7516,amarelo,26
2,Meio-campo,5.0,Bragantino,53,Jadsom,7655,cartao amarelo,1
3,NaN,15.0,Fluminense,15,Douglas Moreira,6262,Amarelo,14
4,NaN,16.0,Avai,90+1,Éverton Silva,5125,AMARELO,14
...,...,...,...,...,...,...,...,...
22833,Zagueiro,21.0,Fluminense,54,Igor Julião,7048,amarelo,17
22834,NAN,17.0,Parana,72,Andrey Nunes dos Santos,6408,cartao amarelo,29
22835,Zagueiro,6.0,Figueirense,26,Marquinhos Pedroso,5705,Vermelho,34
22836,N/A,15.0,Vitoria,9,Luiz Gustavo,4797,amarelo,19


In [2]:
#limpar dados nulos disfarçados
fake_nulls = ["", " ", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "none"]
df.replace(fake_nulls, np.nan, inplace=True)

,posicao,num_camisa,clube,minuto,atleta,partida_id,cartao,rodata
0,Zagueiro,13.0,Ceara,78,Luiz Otávio,7034,cartao amarelo,15
1,Meio-campo,5.0,ATLETICO-GO,8,Willian Maranhão,7516,amarelo,26
2,Meio-campo,5.0,Bragantino,53,Jadsom,7655,cartao amarelo,1
3,NaN,15.0,Fluminense,15,Douglas Moreira,6262,Amarelo,14
4,NaN,16.0,Avai,90+1,Éverton Silva,5125,AMARELO,14
...,...,...,...,...,...,...,...,...
22833,Zagueiro,21.0,Fluminense,54,Igor Julião,7048,amarelo,17
22834,NAN,17.0,Parana,72,Andrey Nunes dos Santos,6408,cartao amarelo,29
22835,Zagueiro,6.0,Figueirense,26,Marquinhos Pedroso,5705,Vermelho,34
22836,N/A,15.0,Vitoria,9,Luiz Gustavo,4797,amarelo,19


In [3]:
#limpar espaços em branco
colunas_texto = df.select_dtypes(include="object").columns
for coluna in colunas_texto:
    df[coluna] = df[coluna].str.strip()


C:\Users\Matheus\AppData\Local\Temp\ipykernel_30340\1325259086.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include="object").columns


In [4]:
#correção do encoding para os nomes
def corrigir_encoding(valor):
    if not isinstance(valor, str):
        return valor
    try:
        return valor.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return valor
 
for coluna in ["atleta", "clube"]:
    df[coluna] = df[coluna].apply(corrigir_encoding)
 
df["clube"] = df["clube"].str.title()
df["atleta"] = df["atleta"].str.title()

In [5]:
#padronizar cartões e  posição
mapa_cartao = {
    "amarelo": "Amarelo", "Amarelo": "Amarelo", "AMARELO": "Amarelo",
    "cartao amarelo": "Amarelo", "Cartão Amarelo": "Amarelo",
    "vermelho": "Vermelho", "Vermelho": "Vermelho", "VERMELHO": "Vermelho",
    "cartao vermelho": "Vermelho", "Cartão Vermelho": "Vermelho",
}

df["cartao"] = df["cartao"].map(mapa_cartao)

mapa_posicao = {
    "zagueiro": "Zagueiro", "Zagueiro": "Zagueiro", "ZAGUEIRO": "Zagueiro",
    "zagueira": "Zagueiro", "Zagueira": "Zagueiro", "ZAGUEIRA": "Zagueiro",
    "meio-campo": "Meio-Campo", "Meio-campo": "Meio-Campo", "MEIO-CAMPO": "Meio-Campo",
    "atacante": "Atacante", "Atacante": "Atacante", "ATACANTE": "Atacante",
    "goleiro": "Goleiro", "Goleiro": "Goleiro", "GOLEIRO": "Goleiro",
}
df["posicao"] = df["posicao"].map(mapa_posicao)


In [6]:
#correção do numero da camisa para um numero inteiro e visto como numero

df["num_camisa"] = pd.to_numeric(df["num_camisa"], errors="coerce")
df["num_camisa"] = df["num_camisa"].astype("Int64")

In [7]:
#correção da tabela minuto igual a feita com a do gol na tabela de gols
df["minuto"] = df["minuto"].astype(object)
tem_acrescimo = df["minuto"].astype(str).str.contains(r"\+", na=False)
partes = df.loc[tem_acrescimo, "minuto"].str.split("+", expand=True)
df.loc[tem_acrescimo, "minuto"] = (
    pd.to_numeric(partes[0], errors="coerce")
    + pd.to_numeric(partes[1], errors="coerce")
)

df["minuto"] = pd.to_numeric(df["minuto"], errors="coerce")

limite_maximo_minuto = 100
outliers_minuto = df["minuto"] > limite_maximo_minuto

df.loc[outliers_minuto, "minuto"] = np.nan

df["minuto"] = df["minuto"].astype("Int64")


In [8]:
#remover linhs duplicadas
chave_unica = ["partida_id","atleta","minuto","cartao"]
duplicatas = df.duplicated(subset=chave_unica).sum()

df = df.drop_duplicates(subset=chave_unica, keep="first")

In [9]:
#reordernação das colunas
ordem_das_colunas =  ["partida_id", "rodata", "minuto", "clube", "atleta","num_camisa", "posicao", "cartao"]

df = df[ordem_das_colunas]
df = df.sort_values(["partida_id","minuto"]).reset_index(drop=True)

In [10]:
#salvar o arqquivo no caminho
print(f"\nLinhas finais: {len(df)}")
print("\nValores nulos restantes por coluna:")
print(df.isna().sum())
print(f"\nFaixa de partida_id nessa tabela: {df['partida_id'].min()} a {df['partida_id'].max()}")
 
print("\n" + "=" * 60)
print("RELATÓRIO DE COBERTURA DE DADOS")
print("=" * 60)
for coluna in ["posicao", "num_camisa"]:
    total_preenchido = df[coluna].notna().sum()
    percentual_cobertura = (total_preenchido / len(df)) * 100
    print(f"{coluna}: {total_preenchido}/{len(df)} linhas ({percentual_cobertura:.1f}%)")
print("=" * 60)



caminho = r"D:\projetos git\analise-de-times-br\data\processed\campeonato-brasileiro-cartoes-limpo.csv"
df.to_csv(caminho, index= False)
display(df)


Linhas finais: 20953

Valores nulos restantes por coluna:
partida_id       0
rodata           0
minuto        2376
clube            0
atleta           6
num_camisa    2437
posicao       3173
cartao           0
dtype: int64

Faixa de partida_id nessa tabela: 4607 a 8785

RELATÓRIO DE COBERTURA DE DADOS
posicao: 17780/20953 linhas (84.9%)
num_camisa: 18516/20953 linhas (88.4%)


,partida_id,rodata,minuto,clube,atleta,num_camisa,posicao,cartao
0,4607,1,66,Figueirense,Paulo Roberto Da Silva,28,NaN,Amarelo
1,4607,1,<NA>,Figueirense,Thiago Heleno,4,Zagueiro,Amarelo
2,4608,1,10,Vitoria,Mansur,<NA>,Zagueiro,Amarelo
3,4608,1,72,Internacional,Andrés D'Alessandro,<NA>,Meio-Campo,Amarelo
4,4608,1,82,Vitoria,Vinícius Santos Silva,<NA>,NaN,Amarelo
...,...,...,...,...,...,...,...,...
20948,8785,38,64,Corinthians,Hugo Souza,1,Goleiro,Amarelo
20949,8785,38,79,Gremio,Douglas Moreira,17,Meio-Campo,Vermelho
20950,8785,38,79,Corinthians,José Andrés Martínez,70,Meio-Campo,Vermelho
20951,8785,38,92,Gremio,Miguel Monsalve,11,Meio-Campo,Amarelo
